# Algorithmic Beauty of Plants (ABOP) Demonstrations

This notebook visualizes the mathematical models and L-Systems based on the book *The Algorithmic Beauty of Plants*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.collections import LineCollection
import sys

# Ensure our local generator scripts can be imported
if '.' not in sys.path:
    sys.path.append('.')

# Setup default visualization settings
plt.style.use('dark_background')


## 1. Iterated Function Systems (Chapter 8)

Visualizing the classic Barnsley Fern using the Chaos Game algorithm stochastic generation.

In [ ]:
import ifs

# Generate the Barnsley fern
fern = ifs.create_barnsley_fern()
pts = fern.generate_chaos_game(iterations=100000)

# Convert list of numpy arrays to a 2D numpy array
pts_array = np.array(pts)
x = pts_array[:, 0]
y = pts_array[:, 1]

# Plot
plt.figure(figsize=(8, 10))
plt.scatter(x, y, s=0.05, c='limegreen', alpha=0.5, marker='.')
plt.title("Barnsley Fern via Chaos Game")
plt.axis('equal')
plt.axis('off')
plt.show()


## 2. Map L-Systems (Chapter 7)

Visualizing the topological relationships (Double-Connected Edge List / Half-Edges) representing cellular division.

In [ ]:
from map_lsystem import MapLSystem

# Initialize and generate map L-system graph
mz = MapLSystem()
v1 = mz.graph.add_vertex([0, 0, 0])
v2 = mz.graph.add_vertex([1, 0, 0])
v3 = mz.graph.add_vertex([0.5, 1, 0])

mz.graph.add_edge(v1, v2, "C1")
mz.graph.add_edge(v2, v3, "C1")
mz.graph.add_edge(v3, v1, "C1")

mz.step()
mz.relax_layout(iterations=50)

# Extract line segments
segments = []
visited = set()
v_keys = list(mz.graph.vertices.keys())
v_map = {vid: mz.graph.vertices[vid].pos[:2] for vid in v_keys}

for he_id, he in mz.graph.half_edges.items():
    if he.id in visited or he.twin.id in visited:
        continue
    p1 = v_map[he.origin]
    p2 = v_map[he.twin.origin]
    segments.append([p1, p2])
    visited.add(he.id)
    visited.add(he.twin.id)

# Plot constraints map
fig, ax = plt.subplots(figsize=(8, 8))
lc = LineCollection(segments, colors='cyan', linewidths=2)
ax.add_collection(lc)

# Scatter exact vertices on top
all_pts = np.array(list(v_map.values()))
ax.scatter(all_pts[:, 0], all_pts[:, 1], c='white', s=50, zorder=5)

ax.set_title("Cell Division Map Graph (Spring-Relaxed)")
ax.axis('equal')
ax.axis('off')
plt.show()


## 3. Phyllotaxis (Chapter 4)

Modeling the distribution of primordia (e.g. sunflower florets, pinecone scales) using Vogel's formula and the golden angle.

In [ ]:
import phyllotaxis

# 1. Planar (Sunflower)
planar_pts = phyllotaxis.generate_planar_phyllotaxis(n_elements=600, c=0.8, alpha=137.5)
planar_arr = np.array(planar_pts)

# 2. Cylindrical (Pinecone / Stem)
cylindrical_pts = phyllotaxis.generate_cylindrical_phyllotaxis(n_elements=400, r=2.0, h=0.1, alpha=137.5)
cyl_arr = np.array(cylindrical_pts)


fig = plt.figure(figsize=(16, 7))

# Planar Subplot
ax1 = fig.add_subplot(1, 2, 1)
# Calculate distance from center for a cool colormap
radii = np.sqrt(planar_arr[:, 0]**2 + planar_arr[:, 1]**2)
scatter1 = ax1.scatter(planar_arr[:, 0], planar_arr[:, 1], c=radii, cmap='Wistia', s=40, edgecolors='k')
ax1.set_title("Planar Phyllotaxis (Sunflower)")
ax1.axis('equal')
ax1.axis('off')

# Cylindrical Subplot
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
# Use height (Y-axis in cyl_arr, which corresponds to index 1) for colormap
ax2.scatter(cyl_arr[:, 0], cyl_arr[:, 1], cyl_arr[:, 2], c=cyl_arr[:, 1], cmap='summer', s=30, depthshade=True)
ax2.set_title("Cylindrical Phyllotaxis (Stem/Pinecone)")
ax2.set_axis_off()

plt.tight_layout()
plt.show()
